 ### Assignment 6: Graph Neural Networks for Traffic Speed Prediction (METR-LA)
 ### Barbod Alamian

### 1. Import Required Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py
import pickle
import time
import os
from sklearn.preprocessing import StandardScaler

from torch_geometric.nn import GCNConv

import warnings
warnings.filterwarnings('ignore')

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Using device: cpu


### 2. Load and Explore Data

In [ ]:
# Set path to your data folder
DATA_PATH = "G:/BA.BA/KNTU - Master/پیله - داده کاوی/TA/A6_GNN/A6_GNN/"

print("Loading data files...")
print("="*50)

# Load sensor IDs
with open(DATA_PATH + "graph_sensor_ids.txt", "r") as f:
    sensor_ids = f.read().strip().split(',')
sensor_ids = [id.strip() for id in sensor_ids]
print(f"Number of sensors: {len(sensor_ids)}")

# Load distance matrix
dist_df = pd.read_csv(DATA_PATH + "distances_la_2012.csv")
print(f"Distance matrix shape: {dist_df.shape}")

# Load adjacency matrix (pickle)
with open(DATA_PATH + "adj_mx.pkl", "rb") as f:
    loaded_data = pickle.load(f, encoding='latin1')

if isinstance(loaded_data, list) and len(loaded_data) >= 3:
    adj_mx = loaded_data[2]
else:
    adj_mx = loaded_data

print(f"Adjacency matrix shape: {adj_mx.shape}")

# Load traffic speed data (HDF5)
with h5py.File(DATA_PATH + "metr-la.h5", "r") as f:
    print("\nHDF5 file structure:")
    print("Keys:", list(f.keys()))
    
    group = f['df']
    print("Keys inside 'df':", list(group.keys()))
    
    traffic_data = None
    for key in group.keys():
        dataset = group[key]
        if hasattr(dataset, 'shape') and len(dataset.shape) >= 2:
            traffic_data = dataset[:]
            print(f"Using dataset: '{key}' with shape {traffic_data.shape}")
            break
    
    if traffic_data is None:
        raise ValueError("No suitable dataset found in the HDF5 file.")

print(f"Traffic data shape: {traffic_data.shape}")


Loading data files...
Number of sensors: 207
Distance matrix shape: (295374, 3)
Adjacency matrix shape: (207, 207)

HDF5 file structure:
Keys: ['df']
Keys inside 'df': ['axis0', 'axis1', 'block0_items', 'block0_values']
Using dataset: 'block0_values' with shape (34272, 207)
Traffic data shape: (34272, 207)


### 3. Data Preprocessing

In [ ]:
print("\nPreprocessing data...")
print("="*50)

# Extract speed data
if traffic_data.ndim == 3:
    speed_data = traffic_data[:, :, 0]
else:
    speed_data = traffic_data

print(f"Speed data shape: {speed_data.shape}")

# Normalize data (per sensor)
scaler = StandardScaler()
speed_scaled = scaler.fit_transform(speed_data.T).T

# Split into train/val/test (70/10/20%)
num_time_steps = speed_scaled.shape[0]
train_size = int(0.7 * num_time_steps)
val_size = int(0.1 * num_time_steps)

train_data = speed_scaled[:train_size]
val_data = speed_scaled[train_size:train_size+val_size]
test_data = speed_scaled[train_size+val_size:]

print(f"Train: {train_data.shape}, Val: {val_data.shape}, Test: {test_data.shape}")

# Create sliding window sequences
HISTORY_LEN = 12
PRED_LEN = 1

def create_sequences(data, history_len=HISTORY_LEN, pred_len=PRED_LEN):
    """Create sequences for time-series prediction"""
    X, y = [], []
    for i in range(len(data) - history_len - pred_len + 1):
        X.append(data[i:i+history_len])
        y.append(data[i+history_len:i+history_len+pred_len].mean(axis=0))
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_data, HISTORY_LEN, PRED_LEN)
X_val, y_val = create_sequences(val_data, HISTORY_LEN, PRED_LEN)
X_test, y_test = create_sequences(test_data, HISTORY_LEN, PRED_LEN)

print(f"Train samples: {X_train.shape[0]}")
print(f"Val samples: {X_val.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")

# Get dimensions for model
num_sensors = X_train.shape[2]  # number of sensors (207)
history_len = X_train.shape[1]   # history length (12)
mlp_input_dim = num_sensors * history_len

print(f"Number of sensors: {num_sensors}")
print(f"History length: {history_len}")
print(f"MLP input dimension: {mlp_input_dim}")



Preprocessing data...
Speed data shape: (34272, 207)
Train: (23990, 207), Val: (3427, 207), Test: (6855, 207)
Train samples: 23978
Val samples: 3415
Test samples: 6843
Number of sensors: 207
History length: 12
MLP input dimension: 2484


### 4. Build Graph (Unweighted and Weighted)

In [ ]:
print("\nBuilding graph...")
print("="*50)

# Unweighted graph
edge_index_unweighted = []
for i in range(adj_mx.shape[0]):
    for j in range(adj_mx.shape[1]):
        if i != j and adj_mx[i, j] > 0:
            edge_index_unweighted.append([i, j])
edge_index_unweighted = torch.tensor(edge_index_unweighted, dtype=torch.long).T
print(f"Unweighted edges: {edge_index_unweighted.shape[1]}")

# Weighted graph (distance-based)
edge_index_weighted = []
edge_weight = []
for i in range(adj_mx.shape[0]):
    for j in range(adj_mx.shape[1]):
        if i != j and adj_mx[i, j] > 0:
            edge_index_weighted.append([i, j])
            edge_weight.append(adj_mx[i, j])
edge_index_weighted = torch.tensor(edge_index_weighted, dtype=torch.long).T
edge_weight = torch.tensor(edge_weight, dtype=torch.float)
print(f"Weighted edges: {edge_index_weighted.shape[1]}")



Building graph...
Unweighted edges: 1515
Weighted edges: 1515


### 5. Model Definitions

In [ ]:
class GCN(torch.nn.Module):
    """GCN Model - Fixed for traffic prediction"""
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers=2):
        super().__init__()
        self.num_layers = num_layers
        
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.convs = torch.nn.ModuleList()
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(hidden_channels, hidden_channels))
        self.conv_last = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index, edge_weight=None):
        x = self.conv1(x, edge_index, edge_weight)
        x = F.relu(x)
        x = F.dropout(x, training=self.training, p=0.1)
        
        for conv in self.convs:
            x = conv(x, edge_index, edge_weight)
            x = F.relu(x)
            x = F.dropout(x, training=self.training, p=0.1)
        
        x = self.conv_last(x, edge_index, edge_weight)
        return x  # shape: (num_sensors, 1)

class MLP(torch.nn.Module):
    """MLP Model (Non-graph baseline)"""
    def __init__(self, in_dim, hidden_dim, out_dim, num_layers=2):
        super().__init__()
        layers = []
        layers.append(nn.Linear(in_dim, hidden_dim))
        layers.append(nn.ReLU())
        for _ in range(num_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.ReLU())
        layers.append(nn.Linear(hidden_dim, out_dim))
        self.mlp = nn.Sequential(*layers)

    def forward(self, x):
        return self.mlp(x)


### 6. Training Function

In [ ]:
def train_model(model, X_train, y_train, X_val, y_val, 
                edge_index=None, edge_weight=None,
                batch_size=256, epochs=20, lr=0.001, 
                is_gcn=True, verbose=True):
    
    X_train_t = torch.tensor(X_train, dtype=torch.float)
    y_train_t = torch.tensor(y_train, dtype=torch.float)
    
    dataset = TensorDataset(X_train_t, y_train_t)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    model = model.to(device)
    if is_gcn and edge_index is not None:
        edge_index = edge_index.to(device)
        if edge_weight is not None:
            edge_weight = edge_weight.to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    loss_fn = torch.nn.MSELoss()
    
    history = {'train_loss': [], 'val_loss': [], 'val_mae': []}
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        num_batches = 0
        
        for batch_X, batch_y in loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)
            
            batch_pred = []
            for i in range(batch_X.shape[0]):
                x = batch_X[i].T  # shape: (num_sensors, history_len)
                
                if is_gcn:
                    out = model(x, edge_index, edge_weight)  # shape: (num_sensors, 1)
                    pred = out.squeeze(1)  # shape: (num_sensors,)
                else:
                    flattened = x.T.reshape(1, -1)  # shape: (1, num_sensors * history_len)
                    out = model(flattened)  # shape: (1, num_sensors)
                    pred = out.squeeze(0)   # shape: (num_sensors,)
                
                batch_pred.append(pred)
            
            batch_pred = torch.stack(batch_pred)  # shape: (batch_size, num_sensors)
            loss = loss_fn(batch_pred, batch_y)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            num_batches += 1
        
        avg_train_loss = total_loss / num_batches
        
        # Validation
        model.eval()
        val_loss = 0
        val_mae = 0
        with torch.no_grad():
            X_val_t = torch.tensor(X_val, dtype=torch.float).to(device)
            y_val_t = torch.tensor(y_val, dtype=torch.float).to(device)
            
            val_preds = []
            for i in range(X_val_t.shape[0]):
                x = X_val_t[i].T
                
                if is_gcn:
                    out = model(x, edge_index, edge_weight)
                    pred = out.squeeze(1)
                else:
                    flattened = x.T.reshape(1, -1)
                    out = model(flattened)
                    pred = out.squeeze(0)
                
                val_preds.append(pred)
            
            val_preds = torch.stack(val_preds)
            val_loss = loss_fn(val_preds, y_val_t).item()
            val_mae = torch.abs(val_preds - y_val_t).mean().item()
        
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(val_loss)
        history['val_mae'].append(val_mae)
        
        if verbose and (epoch % 5 == 0 or epoch == epochs - 1):
            print(f"Epoch {epoch}: Train Loss={avg_train_loss:.4f}, Val MAE={val_mae:.4f}")
    
    return history


### 7. Evaluation Function

In [ ]:
def evaluate_model(model, X_test, y_test, edge_index=None, edge_weight=None, is_gcn=True):
    model.eval()
    model.to(device)
    if is_gcn and edge_index is not None:
        edge_index = edge_index.to(device)
        if edge_weight is not None:
            edge_weight = edge_weight.to(device)
    
    X_test_t = torch.tensor(X_test, dtype=torch.float).to(device)
    y_test_t = torch.tensor(y_test, dtype=torch.float).to(device)
    
    preds = []
    with torch.no_grad():
        for i in range(X_test_t.shape[0]):
            x = X_test_t[i].T
            
            if is_gcn:
                out = model(x, edge_index, edge_weight)
                pred = out.squeeze(1)
            else:
                flattened = x.T.reshape(1, -1)
                out = model(flattened)
                pred = out.squeeze(0)
            
            preds.append(pred)
    
    preds = torch.stack(preds)
    mae = torch.abs(preds - y_test_t).mean().item()
    rmse = torch.sqrt(((preds - y_test_t)**2).mean()).item()
    return mae, rmse


### 8. Experiment 1: GCN vs MLP vs Historical Average

In [ ]:
print("\n" + "="*60)
print("EXPERIMENT 1: GCN vs Non-Graph Models")
print("="*60)

def historical_average_predict(X_train, X_test):
    avg_speed = X_train.mean(axis=1).mean(axis=0)
    return np.repeat(avg_speed[np.newaxis, :], X_test.shape[0], axis=0)

# Train MLP
print("\nTraining MLP...")
start = time.time()
mlp_model = MLP(mlp_input_dim, 64, num_sensors, num_layers=2)
mlp_history = train_model(
    mlp_model, X_train, y_train, X_val, y_val,
    None, None, batch_size=256, epochs=20, lr=0.001,
    is_gcn=False, verbose=True
)
print(f"MLP training time: {time.time() - start:.2f} seconds")

# Train GCN
print("\nTraining GCN...")
start = time.time()
gcn_model = GCN(history_len, 64, 1, num_layers=2)  # in=12, hidden=64, out=1
gcn_history = train_model(
    gcn_model, X_train, y_train, X_val, y_val,
    edge_index_unweighted, None, batch_size=256, epochs=20, lr=0.001,
    is_gcn=True, verbose=True
)
print(f"GCN training time: {time.time() - start:.2f} seconds")

# Evaluate
mae_gcn, rmse_gcn = evaluate_model(gcn_model, X_test, y_test, edge_index_unweighted, is_gcn=True)
mae_mlp, rmse_mlp = evaluate_model(mlp_model, X_test, y_test, is_gcn=False)

avg_pred = historical_average_predict(X_train, X_test)
mae_hist = np.mean(np.abs(avg_pred - y_test))
rmse_hist = np.sqrt(np.mean((avg_pred - y_test)**2))

print("\nTest Results:")
print(f"Historical Avg - MAE: {mae_hist:.4f}, RMSE: {rmse_hist:.4f}")
print(f"MLP - MAE: {mae_mlp:.4f}, RMSE: {rmse_mlp:.4f}")
print(f"GCN - MAE: {mae_gcn:.4f}, RMSE: {rmse_gcn:.4f}")
print(f"GCN improvement over MLP: {((mae_mlp - mae_gcn)/mae_mlp)*100:.2f}%")



EXPERIMENT 1: GCN vs Non-Graph Models

Training MLP...
Epoch 0: Train Loss=0.5221, Val MAE=0.3907
Epoch 5: Train Loss=0.2115, Val MAE=0.2986
Epoch 10: Train Loss=0.1924, Val MAE=0.2832
Epoch 15: Train Loss=0.1855, Val MAE=0.2809
Epoch 19: Train Loss=0.1816, Val MAE=0.2754
MLP training time: 235.37 seconds

Training GCN...
Epoch 0: Train Loss=0.8112, Val MAE=0.5560
Epoch 5: Train Loss=0.7658, Val MAE=0.5492
Epoch 10: Train Loss=0.7632, Val MAE=0.5461
Epoch 15: Train Loss=0.7623, Val MAE=0.5468
Epoch 19: Train Loss=0.7621, Val MAE=0.5452
GCN training time: 1386.71 seconds

Test Results:
Historical Avg - MAE: 0.5392, RMSE: 0.8495
MLP - MAE: 0.2973, RMSE: 0.5303
GCN - MAE: 0.5264, RMSE: 0.8425
GCN improvement over MLP: -77.06%


### 9. Experiment 2: Effect of GCN Depth

In [ ]:
print("\n" + "="*60)
print("EXPERIMENT 2: Effect of GCN Depth")
print("="*60)

depths = [1, 2, 3, 4]
depth_results = {'MAE': [], 'RMSE': [], 'Cosine_Sim': []}

def compute_cosine_similarity(embeddings):
    if len(embeddings) < 2:
        return 0
    norm = embeddings / embeddings.norm(dim=1, keepdim=True)
    sim_matrix = norm @ norm.T
    mask = ~torch.eye(sim_matrix.shape[0], dtype=torch.bool)
    return sim_matrix[mask].mean().item()

for depth in depths:
    print(f"\nTraining GCN with {depth} layers...")
    model = GCN(history_len, 64, 1, num_layers=depth)
    history = train_model(
        model, X_train, y_train, X_val, y_val,
        edge_index_unweighted, None, batch_size=256, epochs=20, lr=0.001,
        is_gcn=True, verbose=False
    )
    
    mae, rmse = evaluate_model(model, X_test, y_test, edge_index_unweighted, is_gcn=True)
    depth_results['MAE'].append(mae)
    depth_results['RMSE'].append(rmse)
    
    model.eval()
    with torch.no_grad():
        x_sample = torch.tensor(X_val[0], dtype=torch.float).to(device)
        x_sample = x_sample.T
        embeddings = model.conv1(x_sample, edge_index_unweighted.to(device))
        cos_sim = compute_cosine_similarity(embeddings)
        depth_results['Cosine_Sim'].append(cos_sim)
    
    print(f"Depth {depth}: MAE={mae:.4f}, RMSE={rmse:.4f}, Cosine Sim={cos_sim:.4f}")

best_depth = depths[np.argmin(depth_results['MAE'])]
print(f"\nBest depth: {best_depth} layers")



EXPERIMENT 2: Effect of GCN Depth

Training GCN with 1 layers...
Depth 1: MAE=0.5272, RMSE=0.8425, Cosine Sim=0.1889

Training GCN with 2 layers...
Depth 2: MAE=0.5270, RMSE=0.8422, Cosine Sim=0.2868

Training GCN with 3 layers...
Depth 3: MAE=0.5319, RMSE=0.8537, Cosine Sim=0.3425

Training GCN with 4 layers...
Depth 4: MAE=0.5334, RMSE=0.8590, Cosine Sim=0.2949

Best depth: 2 layers


### 10. Experiment 3: Weighted vs Unweighted Graph

In [ ]:
print("\n" + "="*60)
print("EXPERIMENT 3: Weighted vs Unweighted Graph")
print("="*60)

mae_unweighted, rmse_unweighted = evaluate_model(
    gcn_model, X_test, y_test, edge_index_unweighted, is_gcn=True
)
print(f"Unweighted - MAE: {mae_unweighted:.4f}, RMSE: {rmse_unweighted:.4f}")

print("\nTraining GCN with Weighted Graph...")
gcn_weighted = GCN(history_len, 64, 1, num_layers=2)
train_model(
    gcn_weighted, X_train, y_train, X_val, y_val,
    edge_index_weighted, edge_weight, batch_size=256, epochs=20, lr=0.001,
    is_gcn=True, verbose=True
)

mae_weighted, rmse_weighted = evaluate_model(
    gcn_weighted, X_test, y_test, edge_index_weighted, edge_weight, is_gcn=True
)
print(f"Weighted - MAE: {mae_weighted:.4f}, RMSE: {rmse_weighted:.4f}")



EXPERIMENT 3: Weighted vs Unweighted Graph
Unweighted - MAE: 0.5264, RMSE: 0.8425

Training GCN with Weighted Graph...
Epoch 0: Train Loss=0.7502, Val MAE=0.5173
Epoch 5: Train Loss=0.6659, Val MAE=0.5105
Epoch 10: Train Loss=0.6645, Val MAE=0.5108
Epoch 15: Train Loss=0.6636, Val MAE=0.5098
Epoch 19: Train Loss=0.6633, Val MAE=0.5095
Weighted - MAE: 0.4822, RMSE: 0.7812


### 11. Experiment 4: Spatial Error Analysis

In [ ]:
print("\n" + "="*60)
print("EXPERIMENT 4: Spatial Error Analysis")
print("="*60)

def compute_per_sensor_error(model, X_test, y_test, edge_index=None, is_gcn=True):
    model.eval()
    model.to(device)
    if is_gcn and edge_index is not None:
        edge_index = edge_index.to(device)
    
    errors = np.zeros(y_test.shape[1])
    counts = np.zeros(y_test.shape[1])
    
    with torch.no_grad():
        X_test_t = torch.tensor(X_test, dtype=torch.float).to(device)
        y_test_t = torch.tensor(y_test, dtype=torch.float).to(device)
        
        for i in range(X_test_t.shape[0]):
            x = X_test_t[i].T
            
            if is_gcn:
                out = model(x, edge_index)
                pred = out.squeeze(1)
            else:
                flattened = x.T.reshape(1, -1)
                out = model(flattened)
                pred = out.squeeze(0)
            
            y_pred = pred.cpu().numpy()
            y_true = y_test[i]
            errors += np.abs(y_pred - y_true)
            counts += 1
    
    return errors / counts

sensor_errors = compute_per_sensor_error(gcn_model, X_test, y_test, edge_index_unweighted, is_gcn=True)

print("\nSensor Error Analysis:")
print(f"Mean Error: {np.mean(sensor_errors):.4f}")
print(f"Std Error: {np.std(sensor_errors):.4f}")
print(f"Max Error: {np.max(sensor_errors):.4f} (Sensor {np.argmax(sensor_errors)})")
print(f"Min Error: {np.min(sensor_errors):.4f} (Sensor {np.argmin(sensor_errors)})")

threshold_90 = np.percentile(sensor_errors, 90)
high_error_sensors = np.where(sensor_errors > threshold_90)[0]
print(f"High-error sensors (top 10%): {len(high_error_sensors)} sensors")

def compute_node_degree(edge_index, num_nodes):
    degree = torch.zeros(num_nodes)
    for i in range(edge_index.shape[1]):
        degree[edge_index[0, i]] += 1
    return degree.numpy()

node_degrees = compute_node_degree(edge_index_unweighted, len(sensor_errors))
degree_threshold = np.percentile(node_degrees, 75)
central = node_degrees > degree_threshold
peripheral = node_degrees <= degree_threshold

print(f"Central sensors mean error: {np.mean(sensor_errors[central]):.4f}")
print(f"Peripheral sensors mean error: {np.mean(sensor_errors[peripheral]):.4f}")



EXPERIMENT 4: Spatial Error Analysis

Sensor Error Analysis:
Mean Error: 0.5264
Std Error: 0.2597
Max Error: 2.0966 (Sensor 56)
Min Error: 0.1773 (Sensor 201)
High-error sensors (top 10%): 21 sensors
Central sensors mean error: 0.6174
Peripheral sensors mean error: 0.4997


### 12. Final Summary

In [13]:
print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)

print("\n1. GCN vs Non-Graph Models:")
print(f"   GCN - MAE: {mae_gcn:.4f}, RMSE: {rmse_gcn:.4f}")
print(f"   MLP - MAE: {mae_mlp:.4f}, RMSE: {rmse_mlp:.4f}")
print(f"   Historical Avg - MAE: {mae_hist:.4f}, RMSE: {rmse_hist:.4f}")
print(f"   GCN improves over MLP by {((mae_mlp - mae_gcn)/mae_mlp)*100:.2f}%")

print("\n2. GCN Depth Analysis:")
print(f"   Best depth: {best_depth} layers")
print(f"   MAE at best depth: {min(depth_results['MAE']):.4f}")
print("   -> Deeper layers beyond 2-3 cause over-smoothing")

print("\n3. Weighted vs Unweighted Graph:")
print(f"   Unweighted MAE: {mae_unweighted:.4f}")
print(f"   Weighted MAE: {mae_weighted:.4f}")
if mae_weighted < mae_unweighted:
    print(f"   -> Weighted graph improves by {((mae_unweighted - mae_weighted)/mae_unweighted)*100:.2f}%")
else:
    print("   -> Unweighted graph performs better")

print("\n4. Spatial Error Analysis:")
print(f"   Mean sensor error: {np.mean(sensor_errors):.4f}")
print(f"   High-error sensors (top 10%): {len(high_error_sensors)}")
print(f"   Central sensors error: {np.mean(sensor_errors[central]):.4f}")
print(f"   Peripheral sensors error: {np.mean(sensor_errors[peripheral]):.4f}")


FINAL SUMMARY

1. GCN vs Non-Graph Models:
   GCN - MAE: 0.5264, RMSE: 0.8425
   MLP - MAE: 0.2973, RMSE: 0.5303
   Historical Avg - MAE: 0.5392, RMSE: 0.8495
   GCN improves over MLP by -77.06%

2. GCN Depth Analysis:
   Best depth: 2 layers
   MAE at best depth: 0.5270
   -> Deeper layers beyond 2-3 cause over-smoothing

3. Weighted vs Unweighted Graph:
   Unweighted MAE: 0.5264
   Weighted MAE: 0.4822
   -> Weighted graph improves by 8.39%

4. Spatial Error Analysis:
   Mean sensor error: 0.5264
   High-error sensors (top 10%): 21
   Central sensors error: 0.6174
   Peripheral sensors error: 0.4997
